# Experimento 3 v2 — Fine-Tuning LoRA + Optuna (Versao Final com Rodizio Gemini)

**Modelos locais (Optuna completo):**
- Llama 3.1 8B — ja completo (ck3_llama.csv)
- Mistral 7B — ja completo (ck3_mistral.csv)
- Qwen 2.5 7B — ja completo (ck3_qwen.csv)
- Gemma 2 9B — roda do zero neste notebook

**Modelo API:**
- Gemini 3.1 Flash Lite com rodizio entre 4 chaves (googleAI_key_2 ate googleAI_key_5)
- Params fixos (temp=0.1, max_chars=1500) — limitacao do plano gratuito documentada

**Secrets necessarios:** HF_TOKEN, googleAI_key_2, googleAI_key_3, googleAI_key_4, googleAI_key_5

**Upload necessario:** ck3_llama.csv, ck3_mistral.csv, ck3_qwen.csv

Ambiente de execucao -> Alterar tipo -> GPU A100

In [ ]:
# Celula 1 - Instalacao
!pip install -q transformers accelerate bitsandbytes gdown scikit-learn matplotlib peft trl optuna google-genai datasets
print('Instalado!')

In [ ]:
# Celula 2 - Imports e autenticacao
import re, gc, json, time, random, os
import numpy as np
import pandas as pd
import torch
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error,
    cohen_kappa_score, f1_score
)
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, TrainingArguments
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer
from datasets import Dataset
from google.colab import userdata
from google import genai
from google.genai import types
from huggingface_hub import login, HfApi

optuna.logging.set_verbosity(optuna.logging.WARNING)

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

# Rodizio entre 4 chaves Gemini
GEMINI_KEYS = [
    userdata.get('googleAI_key_2'),
    userdata.get('googleAI_key_3'),
    userdata.get('googleAI_key_4'),
    userdata.get('googleAI_key_5'),
]
GEMINI_CLIENTS = [genai.Client(api_key=k) for k in GEMINI_KEYS]
GEMINI_IDX     = [0]  # indice atual do rodizio
GEMINI_SLEEP   = 4.0


def proxima_chave():
    idx = GEMINI_IDX[0]
    GEMINI_IDX[0] = (idx + 1) % len(GEMINI_CLIENTS)
    return GEMINI_CLIENTS[idx]


def chamada_com_retry(fn, max_tentativas=6):
    for tentativa in range(max_tentativas):
        try:
            return fn()
        except Exception as e:
            msg = str(e)
            if '429' in msg or 'rate' in msg.lower() or 'quota' in msg.lower():
                espera = (2 ** tentativa) + random.uniform(0, 2)
                print(f'  Rate limit — aguardando {espera:.1f}s')
                time.sleep(espera)
            else:
                raise
    raise RuntimeError('Falhou apos tentativas')


print(f'Autenticado! {len(GEMINI_CLIENTS)} chaves Gemini carregadas.')

In [ ]:
# Celula 3 - Dataset e K-Fold
import gdown

gdown.download(
    'https://drive.google.com/uc?id=1zg9n7EUDWKfu6t_f3aYDS_QroYXGNGwd',
    'meu_dataset.csv', quiet=False
)
df_enem = pd.read_csv('meu_dataset.csv')


def limpar(texto):
    if pd.isna(texto): return ''
    texto = str(texto).strip("[]'\" ")
    texto = texto.replace('\n', ' ')
    texto = re.sub(r'\[[A-Z/]+\]', '', texto)
    texto = re.sub(r'\{[a-z]+\}', '', texto)
    return re.sub(r'\s+', ' ', texto).strip()


df_enem['essay_limpo'] = df_enem['essay'].apply(limpar)
df_enem = df_enem[df_enem['score'] > 0].reset_index(drop=True)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
df_enem['fold'] = -1
for i, (tr, te) in enumerate(skf.split(df_enem, df_enem['score'])):
    df_enem.loc[te, 'fold'] = i

df_treino = df_enem[df_enem['fold'] != 0].reset_index(drop=True)
df_teste  = df_enem[df_enem['fold'] == 0].reset_index(drop=True)
print(f'Total: {len(df_enem)} | Treino: {len(df_treino)} | Teste: {len(df_teste)}')

In [ ]:
# Celula 4 - Dados de treino e funcoes auxiliares

def formatar_treino(row):
    notas_json = (
        '{"C1": ' + str(row['c1']) +
        ', "C2": ' + str(row['c2']) +
        ', "C3": ' + str(row['c3']) +
        ', "C4": ' + str(row['c4']) +
        ', "C5": ' + str(row['c5']) +
        ', "Nota_Total": ' + str(row['score']) + '}'
    )
    instrucao = (
        'Voce e um avaliador oficial de redacoes do ENEM.\n'
        'Avalie usando a escala: 0, 40, 80, 120, 160 ou 200 por competencia.\n'
        'C1(Norma Culta) C2(Tema/Estrutura) C3(Argumentacao) C4(Coesao) C5(Intervencao) — todas 0-200\n\n'
        'REDACAO:\n' + row['essay_limpo'] + '\n\n'
        'Responda APENAS com o JSON:'
    )
    return {'text': instrucao + '\n' + notas_json}


dados_treino = [formatar_treino(row) for _, row in df_treino.iterrows()]
print(f'Pares de treino: {len(dados_treino)}')


def extrair_notas(resposta):
    try:
        texto = re.sub(r'```json|```', '', str(resposta))
        i = texto.find('{')
        j = texto.rfind('}') + 1
        if i == -1 or j <= i: return None
        dados = json.loads(texto[i:j])
        comps = ['C1', 'C2', 'C3', 'C4', 'C5']
        if all(c in dados for c in comps):
            if max(dados[c] for c in comps) <= 20:
                for c in comps: dados[c] *= 10
            dados['Nota_Total'] = sum(dados[c] for c in comps)
            return dados
        if 'Nota_Total' in dados: return dados
    except: pass
    return None


def extrair_notas_markdown(resposta):
    try:
        texto = str(resposta)
        comps = {}
        for c in ['C1', 'C2', 'C3', 'C4', 'C5']:
            m = re.search(rf'{c}[^:]*:\s*(\d+)', texto)
            if m: comps[c] = int(m.group(1))
        if len(comps) == 5:
            if max(comps.values()) <= 20:
                for c in comps: comps[c] *= 10
            comps['Nota_Total'] = sum(comps.values())
            return comps
        m = re.search(r'(?:Total|Nota\s*Total|Nota\s*Final)[^\d]*(\d{3,4})', texto, re.I)
        if m: return {'Nota_Total': int(m.group(1))}
    except: pass
    return None


def extrair(resposta):
    return extrair_notas(resposta) or extrair_notas_markdown(resposta)


def calcular_metricas(y_true, y_pred, nome):
    yt = np.array(y_true, dtype=float)
    yp = np.array(y_pred, dtype=float)
    mae  = mean_absolute_error(yt, yp)
    rmse = float(np.sqrt(mean_squared_error(yt, yp)))
    def disc(n): return np.clip(np.round(np.array(n) / 40).astype(int), 0, 25)
    try:    qwk = cohen_kappa_score(disc(yt), disc(yp), weights='quadratic')
    except: qwk = float('nan')
    try:    f1  = f1_score(disc(yt), disc(yp), average='weighted', zero_division=0)
    except: f1  = float('nan')
    print(f'\n{"="*55}')
    print(f'METRICAS — {nome}')
    print(f'{"="*55}')
    print(f'  Amostras : {len(yt)}')
    print(f'  MAE      : {mae:.4f}')
    print(f'  RMSE     : {rmse:.4f}')
    print(f'  QWK      : {qwk:.4f}')
    print(f'  F1 Score : {f1:.4f}')
    print(f'{"="*55}')
    return {'modelo': nome, 'mae': mae, 'rmse': rmse, 'qwk': qwk, 'f1': f1, 'n': len(yt)}


print('Dados de treino e funcoes auxiliares carregadas!')

In [ ]:
# Celula 5 - Inferencia local e Gemini com rodizio

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16
)


def carregar_para_treino(nome_modelo):
    print(f'Carregando {nome_modelo} para treino...')
    tok = AutoTokenizer.from_pretrained(nome_modelo, trust_remote_code=True)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    tok.padding_side = 'right'
    m = AutoModelForCausalLM.from_pretrained(
        nome_modelo, quantization_config=bnb_config,
        device_map='auto', trust_remote_code=True
    )
    m.config.use_cache = False
    print('Carregado!')
    return tok, m


def liberar(m, tok):
    del m, tok
    gc.collect()
    torch.cuda.empty_cache()
    print('Memoria GPU liberada.')


PROMPT_INF = (
    'Voce e um avaliador oficial de redacoes do ENEM.\n'
    'Avalie usando a escala: 0, 40, 80, 120, 160 ou 200 por competencia.\n'
    'C1(Norma Culta) C2(Tema/Estrutura) C3(Argumentacao) C4(Coesao) C5(Intervencao) — todas 0-200\n\n'
    'REDACAO:\n'
)


def inf_local(tok, m, redacao, temp=0.1, max_tok=300):
    prompt = PROMPT_INF + redacao + '\n\nResponda APENAS com o JSON:'
    msgs = [{'role': 'user', 'content': prompt}]
    try:
        txt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    except:
        txt = prompt
    inp = tok(txt, return_tensors='pt', truncation=True, max_length=3072).to(m.device)
    with torch.no_grad():
        out = m.generate(
            **inp, max_new_tokens=max_tok,
            temperature=temp, do_sample=True,
            pad_token_id=tok.pad_token_id
        )
    return tok.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True).strip()


def inf_gemini(redacao, temp=0.1, max_tok=300):
    prompt = PROMPT_INF + redacao[:1500] + '\n\nResponda APENAS com o JSON:'
    def fn():
        client = proxima_chave()
        resp = client.models.generate_content(
            model='gemini-3.1-flash-lite',
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=temp,
                max_output_tokens=max_tok,
                response_mime_type='application/json'
            )
        )
        return resp.text
    result = chamada_com_retry(fn)
    time.sleep(GEMINI_SLEEP)
    return result


print('Funcoes de inferencia carregadas!')

In [ ]:
# Celula 6 - Optuna para fine-tuning

def objetivo_optuna_ft(trial, nome_modelo):
    lr        = trial.suggest_float('lr', 1e-5, 5e-4, log=True)
    lora_rank = trial.suggest_categorical('lora_rank', [8, 16, 32])
    batch     = trial.suggest_categorical('batch', [1, 2])
    epochs    = trial.suggest_int('epochs', 1, 3)

    lora_cfg = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=lora_rank, lora_alpha=lora_rank * 2,
        lora_dropout=0.05, bias='none',
        target_modules=['q_proj', 'v_proj']
    )

    tok, m = carregar_para_treino(nome_modelo)
    m = get_peft_model(m, lora_cfg)

    ds = Dataset.from_list(dados_treino[:200]).map(
        lambda x: tok(x['text'], truncation=True, max_length=1024, padding='max_length'),
        remove_columns=['text']
    )

    args = TrainingArguments(
        output_dir=f'./optuna_ft_{trial.number}',
        num_train_epochs=epochs,
        per_device_train_batch_size=batch,
        gradient_accumulation_steps=max(1, 4 // batch),
        learning_rate=lr, bf16=True,
        logging_steps=50, save_strategy='no',
        report_to='none', warmup_steps=5
    )
    SFTTrainer(model=m, train_dataset=ds, args=args, processing_class=tok).train()

    m.eval()
    yp, yt = [], []
    for _, row in df_teste.head(30).iterrows():
        try:
            resp  = inf_local(tok, m, row['essay_limpo'])
            notas = extrair(resp)
            if notas and 'Nota_Total' in notas:
                yp.append(notas['Nota_Total'])
                yt.append(row['score'])
        except: pass

    liberar(m, tok)

    if len(yp) < 5: raise optuna.TrialPruned()
    def disc(n): return np.clip(np.round(np.array(n) / 40).astype(int), 0, 25)
    try:    return cohen_kappa_score(disc(yt), disc(yp), weights='quadratic')
    except: return -1.0


def rodar_optuna_ft(nome_modelo, nome_curto, n_trials=5):
    print(f'Optuna: otimizando {nome_curto} ({n_trials} trials)...')
    study = optuna.create_study(
        direction='maximize',
        pruner=optuna.pruners.MedianPruner(n_startup_trials=2, n_warmup_steps=3)
    )
    study.optimize(
        lambda t: objetivo_optuna_ft(t, nome_modelo),
        n_trials=n_trials, catch=(Exception,)
    )
    print(f'Melhores params {nome_curto}: {study.best_params}')
    return study.best_params


print('Optuna para fine-tuning carregado!')

In [ ]:
# Celula 7 - Fine-tuning completo e avaliacao com checkpoint

def rodar_finetuning(nome_modelo, nome_curto, params):
    lr        = params.get('lr', 2e-4)
    lora_rank = params.get('lora_rank', 16)
    batch     = params.get('batch', 2)
    epochs    = params.get('epochs', 3)

    print(f'\nFine-tuning {nome_curto} — lr={lr:.2e} rank={lora_rank} batch={batch} epochs={epochs}')

    lora_cfg = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=lora_rank, lora_alpha=lora_rank * 2,
        lora_dropout=0.05, bias='none',
        target_modules=['q_proj', 'v_proj']
    )

    tok, m = carregar_para_treino(nome_modelo)
    m = get_peft_model(m, lora_cfg)
    m.print_trainable_parameters()

    ds = Dataset.from_list(dados_treino).map(
        lambda x: tok(x['text'], truncation=True, max_length=2048, padding='max_length'),
        remove_columns=['text']
    )

    args = TrainingArguments(
        output_dir=f'./ft_{nome_curto}',
        num_train_epochs=epochs,
        per_device_train_batch_size=batch,
        gradient_accumulation_steps=max(1, 4 // batch),
        learning_rate=lr, bf16=True,
        logging_steps=100, save_strategy='no',
        report_to='none', warmup_steps=20
    )
    SFTTrainer(model=m, train_dataset=ds, args=args, processing_class=tok).train()
    print(f'Fine-tuning de {nome_curto} concluido!')

    m.eval()
    ckpt_csv  = f'ck3_{nome_curto}.csv'
    if os.path.exists(ckpt_csv):
        df_ck     = pd.read_csv(ckpt_csv)
        ja_feitos = set(df_ck['index_redacao'].tolist())
        print(f'Checkpoint: {len(ja_feitos)} ja avaliadas — retomando...')
    else:
        df_ck     = pd.DataFrame(columns=['index_redacao', 'score', 'pred_total'])
        ja_feitos = set()

    pendentes = df_teste[~df_teste.index.isin(ja_feitos)]
    print(f'Pendentes: {len(pendentes)}/{len(df_teste)}')

    novos = []
    for idx, (i, row) in enumerate(pendentes.iterrows()):
        try:
            resp  = inf_local(tok, m, row['essay_limpo'])
            notas = extrair(resp)
            pred  = notas['Nota_Total'] if notas else None
            novos.append({'index_redacao': i, 'score': row['score'], 'pred_total': pred})
        except:
            novos.append({'index_redacao': i, 'score': row['score'], 'pred_total': None})

        if (idx + 1) % 50 == 0:
            df_ck = pd.concat([df_ck, pd.DataFrame(novos)], ignore_index=True)
            df_ck.to_csv(ckpt_csv, index=False)
            novos = []
            print(f'  Checkpoint: {idx + 1 + len(ja_feitos)}/{len(df_teste)}')

    if novos:
        df_ck = pd.concat([df_ck, pd.DataFrame(novos)], ignore_index=True)
        df_ck.to_csv(ckpt_csv, index=False)

    try:
        api   = HfApi()
        repo  = 'YurinhoMatsumoto/enem-' + nome_curto + '-lora-exp3v2'
        pasta = './modelo_ft_' + nome_curto
        m.save_pretrained(pasta)
        tok.save_pretrained(pasta)
        api.create_repo(repo_id=repo, token=HF_TOKEN, private=True, exist_ok=True)
        api.upload_folder(folder_path=pasta, repo_id=repo, token=HF_TOKEN)
        print(f'Modelo salvo no HF Hub: {repo}')
    except Exception as e:
        print(f'Erro ao salvar no HF Hub: {e}')

    liberar(m, tok)

    df_v = df_ck.dropna(subset=['pred_total'])
    print(f'Validas: {len(df_v)}/{len(df_teste)}')
    if len(df_v) > 0:
        return calcular_metricas(
            df_v['score'].tolist(), df_v['pred_total'].tolist(),
            nome_curto + ' (Fine-tuned)'
        )
    return None


print('Funcao de fine-tuning carregada!')

## Recuperar resultados dos modelos ja completos

In [ ]:
# Celula 8 - Recuperar metricas dos modelos ja completos

resultados_anteriores = {}

for nome_curto, nome_display in [
    ('llama',   'Llama 3.1 8B (Fine-tuned)'),
    ('mistral', 'Mistral 7B (Fine-tuned)'),
    ('qwen',    'Qwen 2.5 7B (Fine-tuned)'),
]:
    ckpt = f'ck3_{nome_curto}.csv'
    if os.path.exists(ckpt):
        df_ck = pd.read_csv(ckpt)
        df_v  = df_ck.dropna(subset=['pred_total'])
        if len(df_v) > 0:
            resultados_anteriores[nome_display] = calcular_metricas(
                df_v['score'].tolist(), df_v['pred_total'].tolist(), nome_display
            )
    else:
        print(f'ATENCAO: {ckpt} nao encontrado.')

print(f'\nResultados recuperados: {len(resultados_anteriores)}/3')

## Gemma 2 9B — Optuna + Fine-Tuning

In [ ]:
# Celula 9 - Gemma 2 9B
params_gemma = rodar_optuna_ft('google/gemma-2-9b-it', 'gemma', n_trials=5)
res_gemma    = rodar_finetuning('google/gemma-2-9b-it', 'gemma', params_gemma)

## Gemini 3.1 Flash Lite — Rodizio entre 4 chaves

Params fixos. Rodizio automatico entre googleAI_key_2 ate googleAI_key_5.

In [ ]:
# Celula 10 - Gemini com rodizio de chaves (params fixos)

ckpt_csv = 'ck3_gem.csv'
if os.path.exists(ckpt_csv):
    df_ck_gem = pd.read_csv(ckpt_csv)
    ja_gem    = set(df_ck_gem['index_redacao'].tolist())
    print(f'Checkpoint: {len(ja_gem)} ja feitas')
else:
    df_ck_gem = pd.DataFrame(columns=['index_redacao', 'score', 'pred_total'])
    ja_gem    = set()

pendentes_gem = df_teste[~df_teste.index.isin(ja_gem)]
print(f'Pendentes: {len(pendentes_gem)}/{len(df_teste)}')

novos_gem = []
for idx, (i, row) in enumerate(pendentes_gem.iterrows()):
    try:
        resp  = inf_gemini(row['essay_limpo'])
        notas = extrair(resp)
        pred  = notas['Nota_Total'] if notas else None
        novos_gem.append({'index_redacao': i, 'score': row['score'], 'pred_total': pred})
    except:
        novos_gem.append({'index_redacao': i, 'score': row['score'], 'pred_total': None})

    if (idx + 1) % 50 == 0:
        df_ck_gem = pd.concat([df_ck_gem, pd.DataFrame(novos_gem)], ignore_index=True)
        df_ck_gem.to_csv(ckpt_csv, index=False)
        novos_gem = []
        chave_atual = GEMINI_IDX[0] + 2
        print(f'  Checkpoint: {idx + 1 + len(ja_gem)}/{len(df_teste)} (chave atual: googleAI_key_{chave_atual})')

if novos_gem:
    df_ck_gem = pd.concat([df_ck_gem, pd.DataFrame(novos_gem)], ignore_index=True)
    df_ck_gem.to_csv(ckpt_csv, index=False)

df_v_gem = df_ck_gem.dropna(subset=['pred_total'])
res_gem  = calcular_metricas(
    df_v_gem['score'].tolist(), df_v_gem['pred_total'].tolist(),
    'Gemini 3.1 Flash Lite (Fine-tuning ref)'
) if len(df_v_gem) > 0 else None
print(f'Gemini: {len(df_v_gem)}/{len(df_teste)} validas')

## Resultados Consolidados — Experimento 3

In [ ]:
# Celula 11 - Consolidacao final
import matplotlib.pyplot as plt

novos = [res_gemma, res_gem]
todos = list(resultados_anteriores.values()) + [r for r in novos if r is not None]
df_res = pd.DataFrame(todos).sort_values('qwk', ascending=False).reset_index(drop=True)

print('\n' + '='*65)
print(f'{"Modelo":<32} {"N":>6} {"MAE":>8} {"RMSE":>8} {"QWK":>8} {"F1":>8}')
print('-'*65)
for _, row in df_res.iterrows():
    print(f'{row["modelo"]:<32} {int(row["n"]):>6} {row["mae"]:>8.4f} {row["rmse"]:>8.4f} {row["qwk"]:>8.4f} {row["f1"]:>8.4f}')
print('='*65)

df_res.to_csv('resultados_exp3v2_final.csv', index=False)
print('\nCSV salvo: resultados_exp3v2_final.csv')

In [ ]:
# Celula 12 - Graficos
cores = ['#4C72B0','#DD8452','#55A868','#C44E52','#8172B2']

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle(
    'Experimento 3 v2 — Fine-Tuning LoRA + Optuna\nDesempenho por Modelo e Metrica',
    fontsize=14, fontweight='bold'
)

for ax, (metrica, coluna) in zip(axes.flatten(),
        [('MAE (menor = melhor)', 'mae'), ('RMSE (menor = melhor)', 'rmse'),
         ('QWK (maior = melhor)', 'qwk'), ('F1 Score (maior = melhor)', 'f1')]):
    barras = ax.bar(df_res['modelo'], df_res[coluna],
                    color=cores[:len(df_res)], edgecolor='white')
    for b, v in zip(barras, df_res[coluna]):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.005,
                f'{v:.3f}', ha='center', va='bottom', fontsize=8)
    ax.set_title(metrica, fontsize=12, fontweight='bold')
    ax.set_ylabel('Valor')
    ax.set_xticklabels(df_res['modelo'], rotation=20, ha='right', fontsize=8)
    ax.grid(axis='y', alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('grafico_exp3v2_final.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafico salvo: grafico_exp3v2_final.png')